In [1]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

In [3]:
trees_path = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Code\\thesis\\Output\\30GZ1_07_2.5_1.5_3_polygons.gpkg"
grid_path = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\ecostress_pixels_0602T135437.gpkg"

trees = gpd.read_file(trees_path)
grid = gpd.read_file(grid_path)

if trees.crs != grid.crs:
    trees = trees.to_crs(grid.crs)

# 3. Voeg een uniek ID toe aan het grid als dat er nog niet is
if "grid_id" not in grid.columns:
    grid["grid_id"] = grid.index

# 4. Sla het originele oppervlak van de boomkroon-polygonen op
trees["orig_area"] = trees.geometry.area

# 5. Snijd de boomkronen op per gridcel (Intersection)
inter = gpd.overlay(trees, grid, how="intersection")

# 6. Herbereken het volume naar verhouding van het geknipte oppervlak
inter["clipped_area"] = inter.geometry.area
inter["area_ratio"] = inter["clipped_area"] / inter["orig_area"]

# Gebruik jouw kolomnaam 'measured_cv_m3'
inter["volume_in_cell"] = inter["measured_cv_m3"] * inter["area_ratio"]

# 7. Sommeer het boomkroonvolume per gridcel
vol_per_cell = (
    inter.groupby("grid_id")["volume_in_cell"].sum().reset_index()
)

# 8. Koppel het berekende volume terug aan het oorspronkelijke grid (inclusief temperatuur)
final_grid = grid.merge(vol_per_cell, on="grid_id", how="left")

# Vul cellen zonder bomen op met 0 m³ volume
final_grid["volume_in_cell"] = final_grid["volume_in_cell"].fillna(0)

# 9. Sla op als nieuw bestand
output_path = trees_path.replace(".gpkg", "_ecostress.gpkg")
final_grid.to_file(output_path, driver="GPKG")

print("Succesvol gecombineerd! Resultaat opgeslagen in:", output_path)

C:\Users\hoeks\AppData\Local\Temp\ipykernel_9804\2939541195.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  trees["orig_area"] = trees.geometry.area
C:\Users\hoeks\AppData\Local\Temp\ipykernel_9804\2939541195.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  inter["clipped_area"] = inter.geometry.area


Succesvol gecombineerd! Resultaat opgeslagen in: C:\Floor\Documenten\ENV-year-2\Thesis\Code\thesis\Output\30GZ1_07_2.5_1.5_3_polygons_ecostress.gpkg
